In [ ]:
# from transformers import pipeline

# # pipe = pipeline(model="CohereLabs/aya-vision-8b", task="image-text-to-text", device_map="auto")
# pipe = pipeline(model="CohereLabs/aya-vision-8b", task="image-text-to-text", device_map="mps")

# # Format message with the aya-vision chat template
# messages = [
#     {"role": "user",
#      "content": [
#        {"type": "image", "url": "https://media.istockphoto.com/id/458012057/photo/istanbul-turkey.jpg?s=612x612&w=0&k=20&c=qogAOVvkpfUyqLUMr_XJQyq-HkACXyYUSZbKhBlPrxo="},
#         {"type": "text", "text": "Bu resimde hangi anıt gösterilmektedir?"},
#     ]},
#     ]
# outputs = pipe(text=messages, max_new_tokens=300, return_full_text=False)

# print(outputs)


In [ ]:
# import os
# os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"]

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "CohereLabs/aya-vision-8b"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
).to("mps")


In [ ]:
PROMPT_FILE = "../prompts/prompt_VQA_EN_question.txt"

with open(PROMPT_FILE) as f:
    system_prompt = f.read().splitlines()

system_prompt

In [ ]:
import requests
from PIL import Image
from io import BytesIO
import torch
import time

# Download image manually
# url = "https://media.istockphoto.com/id/458012057/photo/istanbul-turkey.jpg"
# url = "https://huggingface.co/datasets/Atnafu/Afri-MCQA?image-viewer=F8B138E0F895A56D2662413A8D98D3D998ECACDD"
# response = requests.get(url)
# image = Image.open(BytesIO(response.content)).convert("RGB")

# read images
image = Image.open("../data/images/9256185495_1.jpg").convert("RGB")

# Your message — use the PIL image directly, not a URL
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": system_prompt}],
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},   # no "url" key, just a placeholder
            # {"type": "text", "text": "Bu resimde hangi anıt gösterilmektedir?"},
            {"type": "text", "text": "What is the name of the building in this image?"},
        ],
    }
]

# Process inputs — pass image separately via images=
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
    return_dict=True,
) #.to("mps")

inputs = {k: v.to("mps") if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}


# Generate
# Note:
# - No system prompt => Took 1m43.4s
# - With system prompt => Took
# Estimated Time for AYA
# => 3 EXP_SETUP x 5 Languages x 2 responses x 70 samples x 1.5 min = 
with torch.inference_mode():
    start_time = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
    )
    end_time = time.time()
    print("Time: {end_time - start_time}")

# Decode
input_len = inputs["input_ids"].shape[-1]
generated = outputs[0][input_len:]
print(processor.decode(generated, skip_special_tokens=True))